# Lab 7 — Large Language Models (LLMs)



## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


In [1]:
!pip -q install transformers sentencepiece torch



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: C:\Users\kemo7\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [2]:
from transformers import pipeline, set_seed

print("Libraries loaded successfully!")


Libraries loaded successfully!


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [3]:
generator = pipeline("text-generation", model="distilgpt2")

prompt = "Natural language processing is"

output = generator(prompt, max_new_tokens=30, do_sample=False)

print(output[0]["generated_text"])

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Natural language processing is a common feature of the language processing process.























# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt = "Translate to French: I love natural language processing."

# Convert text into tokens
output = model.generate(**tokenizer(prompt, return_tensors="pt"))

# Convert tokens back to text
print(tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


C:\Users\kemo7\AppData\Roaming\Python\Python313\site-packages\transformers\generation\utils.py:1625: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Je s'agissant de la technologie de linguage naturelle.


# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [5]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

text = "This NLP lab is easy and interesting."

result = classifier(text)
print(result)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.999778687953949}]


# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

In [6]:
import pandas as pd

task1_answers = [
    ("Sentiment classification",   "Encoder-only",    "BERT, DistilBERT, RoBERTa"),
    ("Machine translation",        "Encoder-decoder", "T5, BART, MarianMT"),
    ("Open-ended text generation", "Decoder-only",    "GPT family (GPT-2, DistilGPT-2)"),
    ("Named Entity Recognition",   "Encoder-only",    "BERT, RoBERTa"),
    ("Text summarization",         "Encoder-decoder", "T5, BART"),
]

task1_df = pd.DataFrame(task1_answers, columns=["Task", "Most suitable model family", "Example models"])
task1_df.index = task1_df.index + 1
task1_df

,Task,Most suitable model family,Example models
1,Sentiment classification,Encoder-only,"BERT, DistilBERT, RoBERTa"
2,Machine translation,Encoder-decoder,"T5, BART, MarianMT"
3,Open-ended text generation,Decoder-only,"GPT family (GPT-2, DistilGPT-2)"
4,Named Entity Recognition,Encoder-only,"BERT, RoBERTa"
5,Text summarization,Encoder-decoder,"T5, BART"


**Explanation:**

1. **Sentiment classification → Encoder-only.** The model only has to *understand* the whole sentence and give it one label. An encoder reads the full input in both directions, which gives the best representation for classification.
2. **Machine translation → Encoder–decoder.** The input is a sentence in one language and the output is a new sentence in another language. The encoder reads the source sentence and the decoder writes the translation.
3. **Open-ended text generation → Decoder-only.** The model continues a prompt by predicting the next token again and again, which is exactly what a decoder-only model is trained to do.
4. **Named Entity Recognition → Encoder-only.** NER gives a label to every token (person, place, organization, ...). It is a token classification task, so the model needs to understand each word using the words before and after it.
5. **Text summarization → Encoder–decoder.** The encoder reads the long text and the decoder writes a new, shorter text from it.

## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [7]:
sentences = [
    "The movie was excellent.",
    "The lecture was difficult to understand.",
    "I enjoyed learning about transformers.",
]

# `classifier` is the DistilBERT sentiment pipeline created in Part C
results = classifier(sentences)

for sentence, result in zip(sentences, results):
    print(f"Sentence: {sentence}")
    print(f"   Predicted label: {result['label']}")
    print(f"   Score          : {result['score']:.4f}")
    print()

Sentence: The movie was excellent.
   Predicted label: POSITIVE
   Score          : 0.9999

Sentence: The lecture was difficult to understand.
   Predicted label: NEGATIVE
   Score          : 0.9997

Sentence: I enjoyed learning about transformers.
   Predicted label: POSITIVE
   Score          : 0.9997



## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


In [8]:
paragraph = (
    "Natural language processing is a field of artificial intelligence that helps computers "
    "understand, interpret and generate human language. It is used in many applications that "
    "people use every day, such as machine translation, chatbots, search engines and spam "
    "filters. In recent years, large language models based on the transformer architecture "
    "have greatly improved the quality of these applications, because they are trained on "
    "huge amounts of text and can be used for many different tasks."
)

prompt = "Write a short summary of the following text: " + paragraph

# `tokenizer` and `model` are the FLAN-T5-small tokenizer and model loaded in Part B

# Convert text into tokens
inputs = tokenizer(prompt, return_tensors="pt")

# Generate the summary
output = model.generate(**inputs, max_new_tokens=40, num_beams=4, no_repeat_ngram_size=3)

# Convert tokens back to text
summary = tokenizer.decode(output[0], skip_special_tokens=True)

print("Original paragraph:")
print(paragraph)
print()
print("Summary:")
print(summary)
print()
print("Number of words in the paragraph:", len(paragraph.split()))
print("Number of words in the summary  :", len(summary.split()))

Original paragraph:
Natural language processing is a field of artificial intelligence that helps computers understand, interpret and generate human language. It is used in many applications that people use every day, such as machine translation, chatbots, search engines and spam filters. In recent years, large language models based on the transformer architecture have greatly improved the quality of these applications, because they are trained on huge amounts of text and can be used for many different tasks.

Summary:
Natural language processing is a field of artificial intelligence that helps computers understand, interpret and generate human language.

Number of words in the paragraph: 75
Number of words in the summary  : 18



## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**


### Answer

A chatbot has to **produce** text, and a decoder-only model is built for exactly this: it predicts the next token from all the tokens that came before it, and repeats this step until the answer is finished. In a conversation, everything that was said so far (the system instructions, the user's messages and the model's earlier replies) is simply placed in one long sequence, and the model continues that sequence with its reply, so it does not need a separate encoder to read the input. This also makes one model very flexible, because answering questions, translating, summarizing and writing code can all be written as "continue this text". An encoder-only model such as BERT cannot do this well, because it is designed to understand and classify text, not to generate it. Finally, next-token prediction can be trained on huge amounts of plain text without labels, which is what makes models like ChatGPT so knowledgeable and fluent.

# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.


### Sampling with `set_seed(42)`

First the sampling cell is run **with** the seed. `do_sample=True` tells the model to pick the next token randomly according to its probabilities instead of always taking the most likely one. Because the seed is set before each run, the two runs give the same text.

In [9]:
prompt = "Natural language processing is"

for run in range(1, 3):
    set_seed(42)
    output = generator(prompt, max_new_tokens=30, do_sample=True)
    print(f"Run {run}: {output[0]['generated_text']!r}")
    print()

[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 1: 'Natural language processing is a technique used to create new semantic information for language processing, i.e., new semantic information for language processing, and in the context of the language'



Run 2: 'Natural language processing is a technique used to create new semantic information for language processing, i.e., new semantic information for language processing, and in the context of the language'



### Sampling without `set_seed(42)`

In [10]:
for run in range(1, 4):
    output = generator(prompt, max_new_tokens=30, do_sample=True)
    print(f"Run {run}: {output[0]['generated_text']!r}")
    print()

[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 1: 'Natural language processing is an area of interest for both students and their peers. We believe that as a group, we have a culture of openness that is based on a common'



[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 2: 'Natural language processing is a skill that many people have experienced as a result of exposure to such language. This is a skill that many people have experienced as a result of exposure'



Run 3: 'Natural language processing is a new phenomenon in the digital age.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n'



### Answer

A language model does not produce one fixed next word. At every step it produces a **probability distribution** over all the tokens in its vocabulary. With `do_sample=True` the next token is **drawn randomly** from this distribution, so a likely token is chosen often but not always. One different token early in the text changes the context for all the tokens after it, so the outputs quickly become completely different, even though the prompt is the same.

`set_seed(42)` fixes the random number generator, so the same "random" choices are made every time and the output is repeated. When the seed is removed, each run makes different random choices. In Part A the output never changed because `do_sample=False` was used (greedy decoding), where the model always takes the most probable token and there is no randomness.

_____________________________________________________________________________________________

# What I Learned from Lab 7

In this lab I learned about **Large Language Models (LLMs)** and the three main families of transformer models, and I used a pre-trained model from each family with the Hugging Face `transformers` library.

### The Three Transformer Families
- **Encoder-only (BERT, RoBERTa, DistilBERT):** reads the whole input and builds a representation of it. It is the best choice for *understanding* tasks such as sentiment classification and Named Entity Recognition.
- **Encoder–decoder (T5, BART):** the encoder reads the input and the decoder writes a new output. It fits tasks that turn one text into another text, such as translation and summarization.
- **Decoder-only (GPT family):** predicts the next token from the previous tokens. It fits open-ended text generation, completion and conversation.

### Using Pre-trained Models
- `pipeline()` is the easiest way to use a model. It downloads the model and does the tokenization, the prediction and the decoding in one call.
- `AutoTokenizer` and `AutoModelForSeq2SeqLM` do the same steps by hand: the tokenizer converts the text into tokens, `model.generate()` produces the output tokens, and `tokenizer.decode()` converts them back to text.
- I did not train anything in this lab. All the models were already trained, which shows how useful pre-trained models are.

### The Tasks
- **Task 1:** I matched each task with its model family: sentiment classification and NER → encoder-only, machine translation and summarization → encoder–decoder, open-ended text generation → decoder-only.
- **Task 2:** The DistilBERT classifier labeled *"The movie was excellent."* as **POSITIVE (0.9999)**, *"The lecture was difficult to understand."* as **NEGATIVE (0.9997)** and *"I enjoyed learning about transformers."* as **POSITIVE (0.9997)**. All three are correct and the model is very confident.
- **Task 3:** FLAN-T5-small summarized a 75-word paragraph about NLP into one sentence. I found that this small model is **sensitive to the prompt and the generation settings**. My first attempt with the prompt `Summarize:` gave "Learn about natural language processing." repeated many times. A clearer instruction with beam search (`num_beams=4`) and `no_repeat_ngram_size=3` gave a correct summary. The summary is mostly the first sentence of the paragraph, so a small model summarizes by selecting more than by rewriting.
- **Reflection:** A decoder-only model is suitable for a chatbot because a chatbot must generate text, and the whole conversation can be given to the model as one sequence that it continues token by token.

### Optional Challenge: Sampling
- With `do_sample=False` (greedy decoding) the model always takes the most probable token, so the output is always the same. In Part A this also made DistilGPT-2 fall into repeating empty lines.
- With `do_sample=True` the next token is drawn randomly from the probability distribution, so the same prompt gives a different text on every run.
- `set_seed(42)` fixes the random choices, so the two runs with the seed gave exactly the same text, and the three runs without it were all different.

**Conclusion:** The three transformer families are built for different jobs: encoders understand, decoders generate, and encoder–decoders transform one text into another. Choosing the right family for the task matters, and small models like DistilGPT-2 and FLAN-T5-small show the idea well but make mistakes that larger models do not.